In [2]:
#Importar Librerías
import pandas as pd
import glob, os, re, unicodedata, time

In [3]:
CARPETA_ENTRADA = "."
NOMBRE_HOJA_DATOS = "BDD_2025"

In [4]:
MESES_GRUPO1 = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio"]
MESES_GRUPO2 = ["Julio", "Agosto", "Octubre", "Noviembre", "Diciembre"] 

In [5]:
ARCHIVO_SALIDA_GRUPO1 = "BDD_2025_ENERO_JUNIO.xlsx"
ARCHIVO_SALIDA_GRUPO2 = "BDD_2025_JULIO_DICIEMBRE.xlsx"

In [6]:
#Quita tildes y pasa el texto a minúsculas
def normaliza(s):
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return s.lower()

In [7]:
#Devuelve la lista de columnas que no tienen ningún dato
def columnas_totalmente_vacias(df):
    return df.columns[df.isna().all()].tolist()


In [8]:
#Busca, dentro de una lista de rutas de archivos
def encontrar_archivo_mes(mes, archivos):
    mes_norm = normaliza(mes)
    candidatos = [a for a in archivos if mes_norm in normaliza(os.path.basename(a))]
    if len(candidatos) == 0:
        return None  
    if len(candidatos) > 1:
        print(f"  Aviso: más de un archivo coincide con '{mes}': "
              f"{[os.path.basename(c) for c in candidatos]}. Se usará el primero.")
    return candidatos[0]

In [9]:
# Busca todos los .xlsx dentro de CARPETA_ENTRADA
archivos = [
    f for f in glob.glob(os.path.join(CARPETA_ENTRADA, "*.xlsx"))
    if not os.path.basename(f).startswith("~$")
]
print(f"Archivos encontrados en '{CARPETA_ENTRADA}': {len(archivos)}")
for a in sorted(archivos):
    print(" -", os.path.basename(a))

Archivos encontrados en '.': 13
 - 1. BDD_Abril_2025_SM.xlsx
 - 1. BDD_Agosto_2025_SM.xlsx
 - 1. BDD_Diciembre_2025_SM.xlsx
 - 1. BDD_Julio_2025_SM.xlsx
 - 1. BDD_Junio_2025_SM.xlsx
 - 1. BDD_Mayo_2025_SM.xlsx
 - 1. BDD_Noviembre_2025._SMxlsx.xlsx
 - 1. BDD_Octubre_2025_SM.xlsx
 - 1._BDD_Enero_2025_SM.xlsx
 - 1._BDD_Febrero_2025_SM.xlsx
 - 1._BDD_Marzo_2025_SM.xlsx
 - BDD_2025_ENERO_JUNIO.xlsx
 - BDD_2025_JULIO_DICIEMBRE.xlsx


In [10]:
#Lee la hoja de datos de cada mes, las une todas en una sola tabla, limpia columnas vacías, y guarda el resultado.
def unir_meses(meses, archivos, archivo_salida):

    print(f"\n=== Uniendo grupo: {archivo_salida} ===")
    dfs = []

    for mes in meses:
        archivo = encontrar_archivo_mes(mes, archivos)
        if archivo is None:
            print(f"  Aviso: no se encontró archivo para '{mes}', se omite.")
            continue

        t0 = time.time()
        try:
            df = pd.read_excel(archivo, sheet_name=NOMBRE_HOJA_DATOS)
        except ValueError:
            print(f"  Aviso: '{os.path.basename(archivo)}' no tiene una hoja llamada "
                  f"'{NOMBRE_HOJA_DATOS}', se omite.")
            continue

        df["MES"] = mes
        df["ARCHIVO_ORIGEN"] = os.path.basename(archivo)

        dfs.append(df)
        print(f"  {mes}: {os.path.basename(archivo)} -> "
              f"{df.shape[0]} filas x {df.shape[1]} columnas ({time.time()-t0:.1f}s)")

    if not dfs:
        print("  No se encontró ningún archivo para este grupo.")
        return None

    df_unificado = pd.concat(dfs, ignore_index=True, sort=False)

    vacias = columnas_totalmente_vacias(df_unificado)
    df_unificado = df_unificado.drop(columns=vacias)

    print(f"  Total filas unidas: {df_unificado.shape[0]}")
    print(f"  Total columnas: {df_unificado.shape[1]} (se quitaron {len(vacias)} columnas 100% vacías)")

    t0 = time.time()
    df_unificado.to_excel(archivo_salida, sheet_name="BDD_UNIFICADO", index=False)
    print(f"  Archivo guardado en: {archivo_salida} ({time.time()-t0:.1f}s)")

    return df_unificado

In [11]:
#Grupo 1 de Enero a Junio
df_grupo1 = unir_meses(MESES_GRUPO1, archivos, ARCHIVO_SALIDA_GRUPO1)
df_grupo1["MES"].value_counts() if df_grupo1 is not None else None


=== Uniendo grupo: BDD_2025_ENERO_JUNIO.xlsx ===
  Aviso: más de un archivo coincide con 'Enero': ['1._BDD_Enero_2025_SM.xlsx', 'BDD_2025_ENERO_JUNIO.xlsx']. Se usará el primero.
  Enero: 1._BDD_Enero_2025_SM.xlsx -> 1544 filas x 438 columnas (6.5s)
  Febrero: 1._BDD_Febrero_2025_SM.xlsx -> 3086 filas x 354 columnas (9.1s)
  Marzo: 1._BDD_Marzo_2025_SM.xlsx -> 4759 filas x 368 columnas (14.6s)
  Abril: 1. BDD_Abril_2025_SM.xlsx -> 6280 filas x 368 columnas (19.1s)
  Mayo: 1. BDD_Mayo_2025_SM.xlsx -> 8047 filas x 365 columnas (24.6s)
  Aviso: más de un archivo coincide con 'Junio': ['1. BDD_Junio_2025_SM.xlsx', 'BDD_2025_ENERO_JUNIO.xlsx']. Se usará el primero.
  Junio: 1. BDD_Junio_2025_SM.xlsx -> 9698 filas x 368 columnas (29.8s)
  Total filas unidas: 33414
  Total columnas: 363 (se quitaron 75 columnas 100% vacías)
  Archivo guardado en: BDD_2025_ENERO_JUNIO.xlsx (248.9s)


MES
Junio      9698
Mayo       8047
Abril      6280
Marzo      4759
Febrero    3086
Enero      1544
Name: count, dtype: int64

In [12]:
#Julio a Diciembre
df_grupo2 = unir_meses(MESES_GRUPO2, archivos, ARCHIVO_SALIDA_GRUPO2)
df_grupo2["MES"].value_counts() if df_grupo2 is not None else None


=== Uniendo grupo: BDD_2025_JULIO_DICIEMBRE.xlsx ===
  Aviso: más de un archivo coincide con 'Julio': ['1. BDD_Julio_2025_SM.xlsx', 'BDD_2025_JULIO_DICIEMBRE.xlsx']. Se usará el primero.
  Julio: 1. BDD_Julio_2025_SM.xlsx -> 11472 filas x 368 columnas (47.4s)
  Agosto: 1. BDD_Agosto_2025_SM.xlsx -> 13340 filas x 368 columnas (54.7s)
  Octubre: 1. BDD_Octubre_2025_SM.xlsx -> 16808 filas x 368 columnas (59.9s)
  Noviembre: 1. BDD_Noviembre_2025._SMxlsx.xlsx -> 18447 filas x 369 columnas (60.2s)
  Aviso: más de un archivo coincide con 'Diciembre': ['1. BDD_Diciembre_2025_SM.xlsx', 'BDD_2025_JULIO_DICIEMBRE.xlsx']. Se usará el primero.
  Diciembre: 1. BDD_Diciembre_2025_SM.xlsx -> 20346 filas x 369 columnas (66.9s)
  Total filas unidas: 80413
  Total columnas: 368 (se quitaron 2 columnas 100% vacías)
  Archivo guardado en: BDD_2025_JULIO_DICIEMBRE.xlsx (630.0s)


MES
Diciembre    20346
Noviembre    18447
Octubre      16808
Agosto       13340
Julio        11472
Name: count, dtype: int64